# HERD and HED annotations in Spyglass: DANDI:001935 (Uchida lab)

Besides their free-text metadata, the converted files carry two kinds of machine-readable annotation. This notebook reads both back out of a Spyglass database.

- **HERD** (HDMF External Resources Data) links values in the file to terms of external ontologies and registries: the species to NCBI Taxonomy, the strain to an RRID, the experimenter to ORCID, the institution to ROR, and the implant regions and pose keypoints to Uberon anatomy terms.
- **HED** (Hierarchical Event Descriptors, through the [ndx-hed](https://www.hedtags.org/ndx-hed/) extension) describes each session's task condition with a controlled vocabulary. The lab's definitions and the HED schema version are stored once per file.

The tables come from two branches of the CatalystNeuro Spyglass fork, merged into the `uchida-integration` branch used here:

- `herd-external-resources` ([catalystneuro/spyglass PR #2](https://github.com/catalystneuro/spyglass/pull/2)): `OntologyTerm`, a catalog of terms shared across sessions, and `ExternalResources`, one row per file, with a `.Reference` part holding one row per HERD reference.
- `hed-annotations` ([catalystneuro/spyglass PR #3](https://github.com/catalystneuro/spyglass/pull/3)): `HedMetadata`, the schema version and definitions per file, and `HedAnnotation`, one row per HED column of an NWB table, with a `.Row` part holding each row's HED string.

These tables mirror the annotations; they do not change the tables that hold the annotated values, such as `Subject` or `BrainRegion`. Joining by value relates the two, as section 1 shows. Unlike the pose and photometry tables, everything here is in the database itself: this notebook opens no NWB file.

**Running it:** this notebook runs in the `spyglass-uchida-lab-env` conda env inside WSL2, created with `../bootstrap_wsl.sh`. The database is the `spyglass-uchida-db` Docker container on port 3307; see `../docker-compose.yml`, `../dj_local_conf.json` and `../spyglass-ingestion-notes.md`.

In [1]:
from pathlib import Path

import datajoint as dj

# Load the connection config BEFORE importing spyglass: spyglass declares its schemas
# at import time and would otherwise connect with default settings (including TLS,
# which the local Docker MySQL doesn't use).
dj.config.load(str(Path("..") / "dj_local_conf.json"))
dj.conn(use_tls=False)

[2026-11-10 10:52:15,350][INFO]: DataJoint is configured from /mnt/c/Users/amtra/CatalystNeuro/uchida-lab-to-nwb/spyglass/dj_local_conf.json


[2026-11-10 10:52:15,360][INFO]: DataJoint 0.14.10 connected to root@127.0.0.1:3307


DataJoint connection (connected) root@127.0.0.1:3307

In [2]:
import pandas as pd
from hed import HedString, load_schema_version
from hed.models import DefinitionDict
from hed.validator import HedValidator
from IPython.display import HTML, display

from spyglass.common import (
    BrainRegion,
    ExternalResources,
    FiberPhotometryConfig,
    HedAnnotation,
    HedMetadata,
    OntologyTerm,
    Session,
    Subject,
    TaskEpoch,
)
from spyglass.position.v1.imported_multicam_pose import ImportedMultiCameraPose

pd.set_option("display.max_colwidth", 80)

sessions = (
    pd.DataFrame(
        (Session.proj("subject_id", "session_id") * TaskEpoch.proj("task_name")).fetch(
            "nwb_file_name", "subject_id", "session_id", "task_name", as_dict=True
        )
    )
    .sort_values(["subject_id", "session_id"])
    .reset_index(drop=True)
)

## 1. HERD: links to external ontologies

### The terms

`OntologyTerm` holds every term any file refers to, once. Each `entity_id` is a compact identifier, `PREFIX:local_id`, and `entity_uri` resolves it. The prefix names the ontology or registry; the list below links each one present to its [Bioregistry](https://bioregistry.io/) page.

In [3]:
terms = OntologyTerm.fetch(format="frame").reset_index()
terms["prefix"] = terms["entity_id"].str.split(":").str[0]
display(terms)

ONTOLOGY_INFO = {
    "NCBITaxon": ("NCBI Taxonomy", "taxonomic identifiers, here the species"),
    "RRID": ("Research Resource Identifiers", "research resources such as animal strains"),
    "orcid": ("ORCID", "researcher identifiers, here the experimenter"),
    "ror": ("Research Organization Registry", "institution identifiers"),
    "UBERON": ("Uberon", "cross-species anatomy, here brain regions and body parts"),
}
items = []
for prefix, n_terms in terms["prefix"].value_counts().sort_index().items():
    name, description = ONTOLOGY_INFO.get(prefix, (prefix, ""))
    url = f"https://bioregistry.io/registry/{prefix.lower()}"
    items.append(f'<li><a href="{url}" target="_blank"><b>{name}</b></a> (<code>{prefix}</code>, {n_terms} terms): {description}</li>')
display(HTML("<ul>" + "".join(items) + "</ul>"))

,entity_id,entity_uri,prefix
0,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116,NCBITaxon
1,orcid:0000-0002-9093-0505,https://orcid.org/0000-0002-9093-0505,orcid
2,ror:03vek6s52,https://ror.org/03vek6s52,ror
3,RRID:RGD_2308852,https://scicrunch.org/resolver/RRID:RGD_2308852,RRID
4,UBERON:0001130,http://purl.obolibrary.org/obo/UBERON_0001130,UBERON
5,UBERON:0001461,http://purl.obolibrary.org/obo/UBERON_0001461,UBERON
6,UBERON:0001464,http://purl.obolibrary.org/obo/UBERON_0001464,UBERON
7,UBERON:0001465,http://purl.obolibrary.org/obo/UBERON_0001465,UBERON
8,UBERON:0001467,http://purl.obolibrary.org/obo/UBERON_0001467,UBERON
9,UBERON:0001691,http://purl.obolibrary.org/obo/UBERON_0001691,UBERON


### References per session

`ExternalResources` has one row per file, with its number of references. A lone file has 29: species, strain, experimenter, institution, the two implant regions and the 23 skeleton keypoints. A social file also annotates the partner's skeleton, so it has 23 more.

In [4]:
references_per_session = sessions.merge(
    ExternalResources.proj("n_references").fetch(format="frame").reset_index(), on="nwb_file_name"
)
references_per_session

,nwb_file_name,subject_id,session_id,task_name,n_references
0,sub-M4_ses-day-1-lone_.nwb,M4,day-1-lone,lone,29
1,sub-M4_ses-day-1-social_.nwb,M4,day-1-social,social,52
2,sub-M4_ses-day-2-lone_.nwb,M4,day-2-lone,lone,29
3,sub-M4_ses-day-2-social_.nwb,M4,day-2-social,social,52
4,sub-M5_ses-day-1-lone_.nwb,M5,day-1-lone,lone,29
5,sub-M5_ses-day-1-social_.nwb,M5,day-1-social,social,52
6,sub-M5_ses-day-2-lone_.nwb,M5,day-2-lone,lone,29
7,sub-M5_ses-day-2-social_.nwb,M5,day-2-social,social,52
8,sub-M7_ses-day-1-lone_.nwb,M7,day-1-lone,lone,29
9,sub-M7_ses-day-1-social_.nwb,M7,day-1-social,social,52


### One session's references

`fetch_references_dataframe()` returns the references of the restricted sessions joined with their terms. Each row names the annotated NWB object (`object_id`, `object_type`), where in it the value sits (`relative_path`, and `field` for a compound dataset; empty in these files), the value itself (`key`) and the term it links to.

In [5]:
example = sessions.iloc[0]
references = (ExternalResources & {"nwb_file_name": example["nwb_file_name"]}).fetch_references_dataframe()
print(f"{example['subject_id']}, {example['session_id']}: {len(references)} references")
references.sort_values("reference_index")[
    ["reference_index", "object_type", "relative_path", "field", "key", "entity_id", "entity_uri"]
].reset_index(drop=True)

M4, day-1-lone: 29 references


,reference_index,object_type,relative_path,field,key,entity_id,entity_uri
0,0,Subject,species,,Rattus norvegicus,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116
1,1,Subject,strain,,Long Evans,RRID:RGD_2308852,https://scicrunch.org/resolver/RRID:RGD_2308852
2,2,VectorData,,,Nucleus Accumbens,UBERON:0001882,http://purl.obolibrary.org/obo/UBERON_0001882
3,3,VectorData,,,Tail of Striatum,UBERON:0002435,http://purl.obolibrary.org/obo/UBERON_0002435
4,4,Skeleton,nodes,,Snout,UBERON:0006333,http://purl.obolibrary.org/obo/UBERON_0006333
5,5,Skeleton,nodes,,EarLeft,UBERON:0001691,http://purl.obolibrary.org/obo/UBERON_0001691
6,6,Skeleton,nodes,,EarRight,UBERON:0001691,http://purl.obolibrary.org/obo/UBERON_0001691
7,7,Skeleton,nodes,,SpineFront,UBERON:0001130,http://purl.obolibrary.org/obo/UBERON_0001130
8,8,Skeleton,nodes,,SpineMiddle,UBERON:0001130,http://purl.obolibrary.org/obo/UBERON_0001130
9,9,Skeleton,nodes,,SpineLow,UBERON:0001130,http://purl.obolibrary.org/obo/UBERON_0001130


### Linking references to Spyglass's own tables

A reference points at a value in the NWB file, and Spyglass's own tables hold many of the same values. Joining on the value relates them. Three examples:

- **Species.** `Subject.species` holds the species name, and the reference with `object_type` `Subject` and `relative_path` `species` links that name to NCBI Taxonomy.
- **Implant regions.** `BrainRegion.region_name` holds the photometry implant sites, and the references with those names link them to Uberon.
- **Pose keypoints.** `ImportedMultiCameraPose.BodyPart` has one row per keypoint, named `PoseEstimationSeries<keypoint>`, and the references on the skeleton's `nodes` link each keypoint name to Uberon.

In [6]:
all_references = ExternalResources().fetch_references_dataframe()

species = (
    pd.DataFrame((Session * Subject).fetch("nwb_file_name", "subject_id", "species", as_dict=True))
    .merge(
        all_references.query("object_type == 'Subject' and relative_path == 'species'")[["nwb_file_name", "key", "entity_id", "entity_uri"]],
        left_on=["nwb_file_name", "species"], right_on=["nwb_file_name", "key"], how="left",
    )
    .drop_duplicates(["subject_id", "species", "entity_id"])[["subject_id", "species", "entity_id", "entity_uri"]]
    .reset_index(drop=True)
)
display(species)

regions = (
    pd.DataFrame((FiberPhotometryConfig * BrainRegion).fetch("nwb_file_name", "region_name", as_dict=True))
    .drop_duplicates()
    .merge(all_references, left_on=["nwb_file_name", "region_name"], right_on=["nwb_file_name", "key"], how="left")
    .groupby(["region_name", "entity_id", "entity_uri"])["nwb_file_name"].nunique()
    .rename("n_sessions")
    .reset_index()
)
display(regions)

body_parts = pd.DataFrame(
    (ImportedMultiCameraPose.BodyPart & {"nwb_file_name": example["nwb_file_name"]}).fetch(
        "nwb_file_name", "pose_estimation_name", "part_name", as_dict=True
    )
)
body_parts["keypoint"] = body_parts["part_name"].str.removeprefix("PoseEstimationSeries")
node_references = all_references.query("object_type == 'Skeleton' and relative_path == 'nodes'")[
    ["nwb_file_name", "key", "entity_id"]
].drop_duplicates()
keypoints = body_parts.merge(
    node_references, left_on=["nwb_file_name", "keypoint"], right_on=["nwb_file_name", "key"], how="left"
)[["keypoint", "entity_id"]]
assert keypoints["entity_id"].notna().all(), "keypoints without a term"
keypoints.groupby("entity_id", dropna=False)["keypoint"].apply(", ".join).reset_index()

,subject_id,species,entity_id,entity_uri
0,M4,Rattus norvegicus,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116
1,M5,Rattus norvegicus,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116
2,M7,Rattus norvegicus,NCBITaxon:10116,http://purl.obolibrary.org/obo/NCBITaxon_10116


,region_name,entity_id,entity_uri,n_sessions
0,Nucleus Accumbens,UBERON:0001882,http://purl.obolibrary.org/obo/UBERON_0001882,12
1,Tail of Striatum,UBERON:0002435,http://purl.obolibrary.org/obo/UBERON_0002435,12


,entity_id,keypoint
0,UBERON:0001130,"SpineFront, SpineLow, SpineMiddle"
1,UBERON:0001461,"ElbowLeft, ElbowRight"
2,UBERON:0001464,"HipLeft, HipRight"
3,UBERON:0001465,"KneeLeft, KneeRight"
4,UBERON:0001467,"ShoulderLeft, ShoulderRight"
5,UBERON:0001691,"EarLeft, EarRight"
6,UBERON:0002387,"FootLeft, FootRight"
7,UBERON:0002398,"HandLeft, HandRight"
8,UBERON:0002415,TailBase
9,UBERON:0004452,"WristLeft, WristRight"


Every keypoint has a term. Several share one: the three spine points link to the same term, and so do the left and right point of each pair.

### Querying sessions by term

Because the terms are database rows, ontology identifiers work as query keys. The query below finds every session whose file refers to the nucleus accumbens (`UBERON:0001882`), whatever the region is called in that file. Every rat in this dataset has the same two implants, so all sessions match; in a database holding several datasets, the same query selects across them.

In [7]:
NUCLEUS_ACCUMBENS = "UBERON:0001882"
matching = (Session & (ExternalResources.Reference & {"entity_id": NUCLEUS_ACCUMBENS})).fetch(
    "subject_id", "session_id", as_dict=True
)
print(f"{len(matching)} sessions refer to {NUCLEUS_ACCUMBENS}:")
for row in matching:
    print(f"  {row['subject_id']}  {row['session_id']}")

12 sessions refer to UBERON:0001882:
  M4  day-1-lone
  M4  day-1-social
  M4  day-2-lone
  M4  day-2-social
  M5  day-1-lone
  M5  day-1-social
  M5  day-2-lone
  M5  day-2-social
  M7  day-1-lone
  M7  day-1-social
  M7  day-2-lone
  M7  day-2-social


### HERD completeness checks

Every session should annotate the species, strain, experimenter, institution and both implant regions, and every keypoint of every skeleton in the file: the subject's, plus the partner's in social sessions.

In [8]:
N_KEYPOINTS = 23
# (object_type, relative_path) of every annotated value
EXPECTED_PATHS = {
    ("Subject", "species"),
    ("Subject", "strain"),
    ("NWBFile", "general/experimenter"),
    ("NWBFile", "general/institution"),
    ("VectorData", ""),  # brain region column of the fiber photometry table
    ("Skeleton", "nodes"),
}
EXPECTED_REGIONS = {"Nucleus Accumbens", "Tail of Striatum"}
N_SKELETONS = {"lone": 1, "social": 2}

for session in sessions.itertuples():
    name = session.nwb_file_name
    refs = all_references[all_references["nwb_file_name"] == name]
    paths = set(zip(refs["object_type"], refs["relative_path"]))
    assert paths == EXPECTED_PATHS, f"{name}: annotated values {sorted(paths)}"
    assert set(refs.loc[refs["object_type"] == "VectorData", "key"]) == EXPECTED_REGIONS, f"{name}: regions"
    n_nodes = (refs["object_type"] == "Skeleton").sum()
    assert n_nodes == N_KEYPOINTS * N_SKELETONS[session.task_name], f"{name}: {n_nodes} keypoint references"

print(f"All HERD checks passed for {len(sessions)} session(s).")

All HERD checks passed for 12 session(s).


## 2. HED: the task condition in a controlled vocabulary

### Schema version and definitions

`HedMetadata` has one row per file with the HED schema version and the lab's definitions. All files use the same ones. Each definition gives a name to a group of tags, so an annotation can say `Def/Lone-condition` instead of repeating the group.

In [9]:
hed_metadata = HedMetadata.fetch(format="frame").reset_index()
print("HED schema versions:", sorted(hed_metadata["hed_schema_version"].unique()))
print("Distinct definition sets:", hed_metadata["definitions"].nunique())
print()

hed_schema = load_schema_version(hed_metadata["hed_schema_version"].iloc[0])
definitions = DefinitionDict(hed_metadata["definitions"].iloc[0], hed_schema)
for name, definition in definitions.defs.items():
    print(f"{name}: {definition.contents}")

HED schema versions: ['8.5.0']
Distinct definition sets: 1

lone-condition: (Indoors,Real-world,Task,(Animal-agent,Awake,Experiment-participant))
social-condition: (Indoors,Real-world,Task,(Animal-agent,Awake,Experiment-participant),(Animal-agent,Experiment-actor))


### The annotations

`HedAnnotation` has one row per HED column found in a file, here the `HED` column of the task table, and `.Row` has one row per table row. Each task table has one row, which describes the session's single task epoch, so joining on the session lines the HED string up with the task name in `TaskEpoch`.

In [10]:
annotations = (
    HedAnnotation().fetch_annotations_dataframe()
    .merge(sessions, on="nwb_file_name")
    .sort_values(["subject_id", "session_id"])
    .reset_index(drop=True)
)
annotations[["subject_id", "session_id", "task_name", "table_name", "column_name", "row_index", "hed_string"]]

,subject_id,session_id,task_name,table_name,column_name,row_index,hed_string
0,M4,day-1-lone,lone,task_table,HED,0,Def/Lone-condition
1,M4,day-1-social,social,task_table,HED,0,Def/Social-condition
2,M4,day-2-lone,lone,task_table,HED,0,Def/Lone-condition
3,M4,day-2-social,social,task_table,HED,0,Def/Social-condition
4,M5,day-1-lone,lone,task_table,HED,0,Def/Lone-condition
5,M5,day-1-social,social,task_table,HED,0,Def/Social-condition
6,M5,day-2-lone,lone,task_table,HED,0,Def/Lone-condition
7,M5,day-2-social,social,task_table,HED,0,Def/Social-condition
8,M7,day-1-lone,lone,task_table,HED,0,Def/Lone-condition
9,M7,day-1-social,social,task_table,HED,0,Def/Social-condition


### Expanding and querying the annotations

[hedtools](https://pypi.org/project/hedtools/), installed with ndx-hed, expands each `Def/` reference into the tags it stands for. Both conditions describe an awake animal taking part in a task, indoors and in the real world. The social condition adds a second animal as an `Experiment-actor`, the partner.

Once expanded, the tags can be queried. Below, a session is classified as social if its annotation has an `Experiment-actor` tag. The result is compared with the task name from `TaskEpoch`.

In [11]:
def expand(hed_string):
    expanded = HedString(hed_string, hed_schema, def_dict=definitions)
    expanded.expand_defs()
    return expanded


annotations["expanded"] = annotations["hed_string"].map(lambda s: str(expand(s)))
annotations["tags"] = annotations["hed_string"].map(lambda s: {str(t) for t in expand(s).get_all_tags()})
annotations["has_experiment_actor"] = annotations["tags"].map(lambda tags: "Experiment-actor" in tags)

for task_name, group in annotations.groupby("task_name"):
    print(f"{task_name}: {group['hed_string'].iloc[0]}")
    print(f"  expands to {group['expanded'].iloc[0]}")
print()
annotations[["subject_id", "session_id", "task_name", "has_experiment_actor"]]

lone: Def/Lone-condition
  expands to (Def-expand/Lone-condition,(Indoors,Real-world,Task,(Animal-agent,Awake,Experiment-participant)))
social: Def/Social-condition
  expands to (Def-expand/Social-condition,(Indoors,Real-world,Task,(Animal-agent,Awake,Experiment-participant),(Animal-agent,Experiment-actor)))



,subject_id,session_id,task_name,has_experiment_actor
0,M4,day-1-lone,lone,False
1,M4,day-1-social,social,True
2,M4,day-2-lone,lone,False
3,M4,day-2-social,social,True
4,M5,day-1-lone,lone,False
5,M5,day-1-social,social,True
6,M5,day-2-lone,lone,False
7,M5,day-2-social,social,True
8,M7,day-1-lone,lone,False
9,M7,day-1-social,social,True


### Validation and checks

`HedValidator` checks each annotation against the schema and the file's definitions; an empty list of issues means it is valid. The checks then assert, for every session, that the file has HED metadata, that the task table's `HED` column was ingested with one string per row, that it validates, and that its tags agree with the task condition.

In [12]:
validator = HedValidator(hed_schema, def_dicts=definitions)
annotations["n_issues"] = annotations["hed_string"].map(
    lambda s: len(validator.validate(HedString(s, hed_schema, def_dict=definitions), allow_placeholders=False))
)

n_rows = HedAnnotation.fetch(format="frame").reset_index()
for session in sessions.itertuples():
    name = session.nwb_file_name
    assert name in set(hed_metadata["nwb_file_name"]), f"{name}: no HedMetadata"
    columns = n_rows[n_rows["nwb_file_name"] == name]
    assert list(zip(columns["table_name"], columns["column_name"])) == [("task_table", "HED")], f"{name}: HED columns"
    rows = annotations[annotations["nwb_file_name"] == name]
    assert len(rows) == columns["n_rows"].iloc[0] == 1, f"{name}: {len(rows)} HED rows"
    assert (rows["n_issues"] == 0).all(), f"{name}: HED validation issues"
    assert (rows["has_experiment_actor"] == (session.task_name == "social")).all(), (
        f"{name}: HED tags disagree with task {session.task_name}"
    )

print(f"All HED checks passed for {len(sessions)} session(s).")

All HED checks passed for 12 session(s).


## Summary

- **HERD:** every file links its species, strain, experimenter, institution, implant regions and pose keypoints to external terms. In Spyglass the terms are shared rows of `OntologyTerm` and each file's references are rows of `ExternalResources.Reference`, so they can be joined to `Subject`, `BrainRegion` and `ImportedMultiCameraPose.BodyPart` by value, and used as query keys across sessions.
- **HED:** every file describes its task condition with a lab definition, `Def/Lone-condition` or `Def/Social-condition`. `HedMetadata` keeps the schema version and definitions, and `HedAnnotation` the strings. With hedtools they expand into tags that can be validated and queried; the tags agree with `TaskEpoch` in every session.